# **Inferring**
In this lab, you will infer sentiment and topics from product reviews and news articles.

## Setup

In [1]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [2]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

def get_completion(prompt, model="gpt-3.5-turbo"):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0, # this is the degree of randomness of the model's output
    )
    return response.choices[0].message.content

## Product review text

In [3]:
lamp_review = """
Needed a nice lamp for my bedroom, and this one had \
additional storage and not too high of a price point. \
Got it fast.  The string to our lamp broke during the \
transit and the company happily sent over a new one. \
Came within a few days as well. It was easy to put \
together.  I had a missing part, so I contacted their \
support and they very quickly got me the missing piece! \
Lumina seems to me to be a great company that cares \
about their customers and products!!
"""

## Sentiment (positive/negative)

In [4]:
prompt = f"""
What is the sentiment of the following product review, 
which is delimited with triple backticks?

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

The sentiment of the review is positive. The reviewer is satisfied with the lamp, the customer service, and the company in general.


In [5]:
prompt = f"""
What is the sentiment of the following product review, 
which is delimited with triple backticks?

Give your answer as a single word, either "positive" \
or "negative".

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

Positive


## Identify types of emotions

In [6]:
prompt = f"""
Identify a list of emotions that the writer of the \
following review is expressing. Include no more than \
five items in the list. Format your answer as a list of \
lower-case words separated by commas.

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

happy, satisfied, grateful, impressed, pleased


## Identify anger

In [7]:
prompt = f"""
Is the writer of the following review expressing anger?\
The review is delimited with triple backticks. \
Give your answer as either yes or no.

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

No


## Extract product and company name from customer reviews

In [8]:
prompt = f"""
Identify the following items from the review text: 
- Item purchased by reviewer
- Company that made the item

The review is delimited with triple backticks. \
Format your response as a JSON object with \
"Item" and "Brand" as the keys. 
If the information isn't present, use "unknown" \
as the value.
Make your response as short as possible.
  
Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

{
    "Item": "lamp",
    "Brand": "Lumina"
}


## Doing multiple tasks at once

In [9]:
prompt = f"""
Identify the following items from the review text: 
- Sentiment (positive or negative)
- Is the reviewer expressing anger? (true or false)
- Item purchased by reviewer
- Company that made the item

The review is delimited with triple backticks. \
Format your response as a JSON object with \
"Sentiment", "Anger", "Item" and "Brand" as the keys.
If the information isn't present, use "unknown" \
as the value.
Make your response as short as possible.
Format the Anger value as a boolean.

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

{
    "Sentiment": "positive",
    "Anger": false,
    "Item": "lamp",
    "Brand": "Lumina"
}


## Inferring Text Topics
Another application inferring by an LLM is deducing topics from a lengthy piece of text.

This time, the sample is regarding a fictitious newspaper article about a survey conducted by the government measuring the satisfaction rate of workers in government agencies. The results reveal that NASA workers had the highest satisfaction rating.Inferring Text Topics
Another application inferring by an LLM is deducing topics from a lengthy piece of text.

This time, the sample is regarding a fictitious newspaper article about a survey conducted by the government measuring the satisfaction rate of workers in government agencies. The results reveal that NASA workers had the highest satisfaction rating.

In [10]:
story = """
In a recent survey conducted by the government, 
public sector employees were asked to rate their level 
of satisfaction with the department they work at. 
The results revealed that NASA was the most popular 
department with a satisfaction rating of 95%.

One NASA employee, John Smith, commented on the findings, 
stating, "I'm not surprised that NASA came out on top. 
It's a great place to work with amazing people and 
incredible opportunities. I'm proud to be a part of 
such an innovative organization."

The results were also welcomed by NASA's management team, 
with Director Tom Johnson stating, "We are thrilled to 
hear that our employees are satisfied with their work at NASA. 
We have a talented and dedicated team who work tirelessly 
to achieve our goals, and it's fantastic to see that their 
hard work is paying off."

The survey also revealed that the 
Social Security Administration had the lowest satisfaction 
rating, with only 45% of employees indicating they were 
satisfied with their job. The government has pledged to 
address the concerns raised by employees in the survey and 
work towards improving job satisfaction across all departments.
"""

Five topics discussed in the article are requested from the model in a format that each item is one or two words long and in a comma-separated list. ChatGPT returns the topics as government surveys, job satisfaction, NASA, etc.

In [11]:
prompt = f"""
Determine five topics that are being discussed in the \
following text, which is delimited by triple backticks.

Make each item one or two words long. 

Format your response as a list of items separated by commas without numbering them.

Text sample: '''{story}'''
"""
response = get_completion(prompt)
print(response)

Government survey, Employee satisfaction, NASA, Social Security Administration, Job satisfaction


In [12]:
response.split(sep=', ')

['Government survey',
 'Employee satisfaction',
 'NASA',
 'Social Security Administration',
 'Job satisfaction']

## Make a news alert for certain topics

The final sample application is about the selection of topics that a text covers, among a targeted topics list. Initially, the list of possible topics is defined:The final sample application is about the selection of topics that a text covers, among a targeted topics list. Initially, the list of possible topics is defined:

In [13]:
topic_list = [
    "nasa", "local government", "engineering", 
    "employee satisfaction", "federal government"
]

In [14]:
prompt = f"""
Determine whether each item in the following list of \
topics is a topic in the text below, which
is delimited with triple backticks.

Give your answer as a dictionay where the key is a topic and the value is 0 or 1 for each topic if it appears.\

List of topics: {", ".join(topic_list)}

Text sample: '''{story}'''
"""
response = get_completion(prompt)
print(response)

{
    "nasa": 1,
    "local government": 0,
    "engineering": 0,
    "employee satisfaction": 1,
    "federal government": 1
}


In [15]:
for i in response.split(', '):
    print(i)

{
    "nasa": 1,
    "local government": 0,
    "engineering": 0,
    "employee satisfaction": 1,
    "federal government": 1
}


In [17]:
import json

topic_dict = json.loads(response)
if topic_dict['nasa'] == 1:
    print("ALERT: New NASA story!")

ALERT: New NASA story!


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

## House of the Dragon

In [18]:
tvseries_review = """
House of the Dragon faced the unfair expectation 
not only of living up to the legacy of Game of Thrones 
but also of redeeming the underwhelming final season of the HBO fantasy series. 
The first season of House of the Dragon compressed a lot into ten episodes, 
introducing the Targaryen power struggle across the twenty-year reign of King Viserys 
and the power vacuum left in his absence. 
The eight-episode second year chronicled the next months in a much slower season 
that felt stalled compared to the breadth of the first run. 
After two years waiting for the third season, 
House of the Dragon returns with the level of action and intensity 
that we have been waiting to see for the past four years. 
With the epic and bloody showdown between the families of Westeros 
delivering on a massive scale, House of the Dragon has 
become the worthy successor to Game of Thrones we have always wanted it to be, 
but it may be too little too late.
"""

In [19]:
prompt = f"""
What is the sentiment of the following tv series review?

Review text: '''{tvseries_review}'''
"""
response = get_completion(prompt)
print(response)

The sentiment of the review is mixed. The reviewer acknowledges the challenges faced by House of the Dragon in living up to the legacy of Game of Thrones and redeeming its final season. They appreciate the level of action and intensity in the third season but also express disappointment that it may be too little too late. Overall, there is a sense of both satisfaction and resignation in the review.


In [20]:
prompt = f"""
Identify a list of emotions that the writer of the \
following review is expressing. Include no more than \
five items in the list. Format your answer as a list of \
lower-case words separated by commas.

Review text: '''{tvseries_review}'''
"""
response = get_completion(prompt)
print(response)

unfair, underwhelming, stalled, epic, intense


In [21]:
prompt = f"""
What is the sentiment of the following tv series review?

Give your answer as a single word, either "positive" \
or "negative".

Review text: '''{tvseries_review}'''
"""
response = get_completion(prompt)
print(response)

Positive


## World Cup match review - Argentina favored

In [22]:
match_review = """
The controversy began when Portuguese referee Joao Pinheiro initially booked Argentina's Leandro Paredes for a challenge on Embolo. However, video assistant referee Guillermo Pacheco Larios recommended an on-field review for "mistaken identity," leading the official to conclude that Embolo had actually dived. The yellow card for Paredes was rescinded and shown to Embolo instead, resulting in a second caution and a red card for the Rennes forward.
Midfielder Remo Freuler was equally scathing of the officiating standards at the Kansas City Stadium. Freuler stated: "It's just a disaster. I don't know what the referee is doing here. I don't understand why they call him for a situation like this because there are many fouls like this in the first half. Maybe he has to call them for yellow card too. I don't understand how can VAR change a game with this situation. Just let the referee do his thing, you know?"

Embolo 'shattered' by emotional exit
Embolo was seen in tears as he left the pitch. The striker had been a key threat for the Swiss, but found himself the victim of a strict application of IFAB's new rule updates for the 2026 tournament. The Switzerland bench exploded in anger when the decision was announced, while Embolo had to be escorted to the dressing room by his teammates.
Yakin offered an insight into the mood in the dressing room, explaining the impact on his star attacker. Yakin said: "You can imagine how he's doing. He is shattered. He couldn't help the team today. It hurts us and it hurts him. It was a referee mistake. There was definitely no reason to award that yellow card. I don't understand it, it was a harmless situation."
"""

In [23]:
prompt = f"""
What is the sentiment of the following match review?

Give your answer as a single word, either "positive" \
or "negative".

Review text: '''{match_review}'''
"""
response = get_completion(prompt)
print(response)

Negative


In [24]:
prompt = f"""
What is the sentiment of the following match review?

Review text: '''{match_review}'''
"""
response = get_completion(prompt)
print(response)

The sentiment of the match review is negative, with criticism towards the referee's decision and the use of VAR. Players and coaches expressed frustration and disbelief at the officiating standards and the impact it had on the game, particularly on Embolo who was shown a red card due to a mistaken identity. The emotional reaction of Embolo and the team further highlight the disappointment and frustration felt towards the referee's decision.


In [25]:
prompt = f"""
Identify a list of emotions that the writer of the \
following review is expressing. Include no more than \
five items in the list. Format your answer as a list of \
lower-case words separated by commas.

Review text: '''{match_review}'''
"""
response = get_completion(prompt)
print(response)

angry, frustrated, devastated, confused, hurt


Used inferrence on 2 different texts like a tv series review and a world cup match review.
OpenAi API performed well and without hallucinations, when inferring sentiment from reviews.